# FINAL — Robust Vietnamese Traffic Sign Detection

**One Colab notebook only.** Designed for the current `MyDrive/DIP/video1.mp4`.

Goal:
- keep nearby/repeated signs stable,
- recover distant signs by **cropping/zooming the road-sign band before inference**,
- avoid false positives with cross-model agreement, color/shape evidence, templates, NMS, and temporal confirmation,
- never draw stale/predicted boxes when the detector has no current evidence.

Important fix: the primary `vtsr.torchscript` model is a **static 640×640 export**. This notebook manually letterboxes every primary input to exactly 640×640; it never changes the TorchScript inference size.

Select **T4 GPU** then use **Runtime → Run all**.


In [ ]:
# 1) Install + mount Drive + persistent model cache
!pip install -q "ultralytics>=8.3,<9" "huggingface_hub>=0.25" "opencv-python-headless>=4.9" "requests>=2.31" "tqdm>=4.66"

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from dataclasses import dataclass, field
from collections import defaultdict, deque
import os, shutil, subprocess, math, re, csv, json
import cv2, numpy as np, torch, requests
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

ROOT = Path('/content/drive/MyDrive/DIP')
VIDEO = ROOT / 'video1.mp4'
MODELS = ROOT / 'models'
OUTPUTS = ROOT / 'outputs'
TMPL = MODELS / 'sign_templates'
P_DIR = MODELS / 'traffic_sign_primary'
S_DIR = MODELS / 'traffic_sign_secondary'

for p in (MODELS, OUTPUTS, TMPL, P_DIR, S_DIR):
    p.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'Enable T4 GPU first: Runtime → Change runtime type → T4 GPU'
assert VIDEO.exists(), f'Missing input video: {VIDEO}'
DEVICE = 0
print('GPU:', torch.cuda.get_device_name(0))
print('Input:', VIDEO)

# Clean obsolete artifacts from previous helmet/plate/OCR versions.
for p in [
    MODELS/'helmet', MODELS/'license_plate', MODELS/'scene', MODELS/'easyocr',
    OUTPUTS/'video1_plates', OUTPUTS/'video1_violations'
]:
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)

for p in [
    MODELS/'helmet_best.pt', MODELS/'plate_best.pt',
    OUTPUTS/'video1_result.mp4', OUTPUTS/'video1_result.csv',
    OUTPUTS/'video1_result_temp.mp4'
]:
    try:
        p.unlink()
    except FileNotFoundError:
        pass

# Persistent weights: download once, reuse after Colab disconnect.
P_PATH = hf_hub_download(
    repo_id='liamxdev/vtsr',
    filename='vtsr.torchscript',
    local_dir=str(P_DIR),
)
S_PATH = hf_hub_download(
    repo_id='star092304/traffic-sign-detection-vietnam-yolo',
    filename='best.pt',
    local_dir=str(S_DIR),
)

primary = YOLO(P_PATH, task='detect')
secondary = YOLO(S_PATH)
print('Primary:', P_PATH)
print('Secondary:', S_PATH)


In [ ]:
# 2) Persistent template bank
# Templates NEVER decide a class by themselves. They only verify a class already proposed by YOLO.

LEGACY = {
    'No Entry': ['camnguocchieu.jpg', 'wrongway.png'],
    'No Parking': ['noparking.png'],
    'No Stop/Parking': ['nostopandparking.png', 'camdungcamdoxe.png'],
    'No Left Turn': ['noleftturn.png', 'noleft.jpg'],
    'Keep Right': ['keepright.png'],
    'Children': ['children.png'],
    'Slow Down': ['slow.png'],
}

# Wikimedia Commons artwork for Vietnamese road signs (best effort; failure is non-fatal).
OFFICIAL = {
    'No Entry': 'Vietnam road sign P102.svg',
    'No Stop/Parking': 'Vietnam road sign P130.svg',
    'No Parking': 'Vietnam road sign P131a.svg',
    'No Left Turn': 'Vietnam road sign P123a.svg',
    'No Right Turn': 'Vietnam road sign P123b.svg',
    'No U-Turn': 'Vietnam road sign P124a1.svg',
    'Keep Right': 'Vietnam road sign R302a.svg',
    'Children': 'Vietnam road sign W225.svg',
    'Road Works': 'Vietnam road sign W227.svg',
    'No Overtaking': 'Vietnam road sign P125.svg',
}

def safe_name(s):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', s).strip('_')

def save_url(url, path):
    if path.exists() and path.stat().st_size > 500:
        return True
    try:
        r = requests.get(url, timeout=25, headers={'User-Agent':'DIP-traffic-sign-project/2.0'})
        r.raise_for_status()
        path.write_bytes(r.content)
        return True
    except Exception as e:
        print('[template warning]', url, '->', e)
        return False

# Existing project templates are copied into persistent Drive cache.
for label, names in LEGACY.items():
    for i, name in enumerate(names):
        suffix = Path(name).suffix
        url = f'https://raw.githubusercontent.com/NVTruong473/DIP/feature/yolo-traffic-safety/END_DIP/sign_templates/{name}'
        save_url(url, TMPL / f'{safe_name(label)}_legacy_{i}{suffix}')

api = 'https://commons.wikimedia.org/w/api.php'
for label, filename in OFFICIAL.items():
    out = TMPL / f'{safe_name(label)}_official.png'
    if out.exists() and out.stat().st_size > 500:
        continue
    try:
        params = {
            'action':'query', 'format':'json', 'prop':'imageinfo',
            'iiprop':'url', 'iiurlwidth':320, 'titles':f'File:{filename}'
        }
        data = requests.get(api, params=params, timeout=25,
                            headers={'User-Agent':'DIP-traffic-sign-project/2.0'}).json()
        page = next(iter(data['query']['pages'].values()))
        info = page['imageinfo'][0]
        save_url(info.get('thumburl') or info['url'], out)
    except Exception as e:
        print('[Commons warning]', label, '->', e)

print('Template cache:', len(list(TMPL.glob('*'))), 'files')
print('Template folder:', TMPL)


In [ ]:
# 3) Core utilities: label normalization, geometry, evidence, static-640 primary inference

PRIMARY_640 = 640

COMMON = {
    'P-102':'No Entry',
    'P-123A':'No Left Turn',
    'P-123B':'No Right Turn',
    'P-124A':'No U-Turn',
    'P-124A1':'No U-Turn',
    'P-125':'No Overtaking',
    'P-127':'Speed Limit',
    'P-130':'No Stop/Parking',
    'P-131A':'No Parking',
    'P-245A':'Slow Down',
    'R-302A':'Keep Right',
    'R-302B':'Keep Left',
    'R-303':'Roundabout',
    'R-407A':'One Way',
    'W-224':'Pedestrian Crossing',
    'W-225':'Children',
    'W-227':'Road Works',
}

SHORT = {
    'No Stopping & No Parking':'No Stop/Parking',
    'Children Crossing':'Children',
    'Road Work Ahead':'Road Works',
    'No U-Turn and No Left Turn':'No U-Turn/Left',
    'No U-Turn and No Right Turn':'No U-Turn/Right',
    'Intersection with a Minor Road':'Minor Junction',
    'Intersection with Equal Roads':'Equal Junction',
    'Intersection with a Priority Road':'Priority Junction',
    'Level Crossing with Barriers':'Rail Crossing',
    'No Two or Three-wheeled Vehicles':'No 2/3-Wheelers',
    'Road with Surveillance Camera':'Camera Ahead',
}
EXCLUDE = {'Green Light', 'Red Light'}

@dataclass
class Det:
    box: tuple
    label: str
    conf: float
    source: str
    family: str
    color: float = 0.0
    shape: float = 0.0
    tmpl: float = 0.0
    score: float = 0.0

def canonical_code(s):
    s = str(s).upper().replace('_','-').replace('.','-').replace(' ','')
    while '--' in s:
        s = s.replace('--','-')
    return s

def primary_label(raw):
    c = canonical_code(raw)
    if c in COMMON:
        return COMMON[c]
    if c.startswith('P-'):
        return f'Prohibition {c}'
    if c.startswith('R-'):
        return f'Mandatory {c}'
    if c.startswith('W-'):
        return f'Warning {c}'
    return f'Traffic Sign {c}'[:30]

def secondary_label(raw):
    s = re.sub(r'\s+', ' ', str(raw)).strip()
    return SHORT.get(s, s[:30])

def iou(a, b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    if inter <= 0:
        return 0.0
    aa=max(1,a[2]-a[0])*max(1,a[3]-a[1])
    bb=max(1,b[2]-b[0])*max(1,b[3]-b[1])
    return inter / (aa + bb - inter)

def center_match(a, b):
    ax=(a[0]+a[2])/2; ay=(a[1]+a[3])/2
    bx=(b[0]+b[2])/2; by=(b[1]+b[3])/2
    dist=math.hypot(ax-bx, ay-by)
    scale=max(10.0, 0.5*((a[2]-a[0])+(a[3]-a[1])+(b[2]-b[0])+(b[3]-b[1]))/2)
    return dist/scale

def plausible(box, W, H):
    w=max(1,box[2]-box[0]); h=max(1,box[3]-box[1])
    ar=w/h
    area=(w*h)/(W*H)
    # permissive enough for circular/triangular/rectangular traffic signs,
    # but reject tiny speckles, extreme banners and giant scene regions.
    return min(w,h) >= 6 and 0.18 <= ar <= 5.5 and area < 0.13

def color_support(crop):
    if crop is None or crop.size == 0:
        return 0.0
    hsv=cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    masks=[
        cv2.inRange(hsv,(0,55,40),(12,255,255)),
        cv2.inRange(hsv,(165,55,40),(180,255,255)),
        cv2.inRange(hsv,(88,50,35),(138,255,255)),
        cv2.inRange(hsv,(14,50,45),(42,255,255)),
    ]
    m=masks[0]
    for z in masks[1:]:
        m=cv2.bitwise_or(m,z)
    return float(np.count_nonzero(m)/m.size)

def shape_support(crop):
    if crop is None or crop.size == 0:
        return 0.0
    g=cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    g=cv2.GaussianBlur(g,(3,3),0)
    e=cv2.Canny(g,55,150)
    cnts,_=cv2.findContours(e,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    H,W=g.shape[:2]
    best=0.0
    for c in cnts:
        a=cv2.contourArea(c)
        if a < 0.08*W*H:
            continue
        per=cv2.arcLength(c,True)
        if per <= 0:
            continue
        circ=4*math.pi*a/(per*per+1e-6)
        n=len(cv2.approxPolyDP(c,0.04*per,True))
        s=max(min(1.0,circ/0.75), 0.8 if 3<=n<=6 else 0.0)
        best=max(best,s)
    return best

def adaptive_enhance(img):
    """Inference-only enhancement. Saved output remains the untouched original frame."""
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    v=hsv[:,:,2]
    mean=float(v.mean()); std=float(v.std())
    x=img
    if mean < 80:  # night / deep shadow
        gamma=1.45
        lut=np.array([((i/255.0)**(1/gamma))*255 for i in range(256)],dtype=np.uint8)
        x=cv2.LUT(x,lut)
    lab=cv2.cvtColor(x,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clip=2.0 if (mean<95 or std<45 or mean>195) else 1.35
    l=cv2.createCLAHE(clipLimit=clip,tileGridSize=(8,8)).apply(l)
    x=cv2.cvtColor(cv2.merge([l,a,b]),cv2.COLOR_LAB2BGR)
    blur=cv2.GaussianBlur(x,(0,0),0.9)
    return cv2.addWeighted(x,1.10,blur,-0.10,0)

def letterbox_square(img, size=640):
    """Exact square input for static TorchScript. Returns square and inverse mapping."""
    h,w=img.shape[:2]
    r=min(size/w, size/h)
    nw=max(1,int(round(w*r))); nh=max(1,int(round(h*r)))
    resized=cv2.resize(img,(nw,nh),interpolation=cv2.INTER_LINEAR)
    dw=size-nw; dh=size-nh
    left=dw//2; right=dw-left; top=dh//2; bottom=dh-top
    square=cv2.copyMakeBorder(resized,top,bottom,left,right,cv2.BORDER_CONSTANT,value=(114,114,114))
    return square, r, left, top

# Edge template verifier.
TF=defaultdict(list)
for p in TMPL.glob('*'):
    img=cv2.imread(str(p))
    if img is None:
        continue
    label=None
    for k in OFFICIAL:
        if p.name.startswith(safe_name(k)):
            label=k; break
    if label is None:
        for k in LEGACY:
            if p.name.startswith(safe_name(k)):
                label=k; break
    if label is None:
        continue
    g=cv2.cvtColor(cv2.resize(img,(72,72)),cv2.COLOR_BGR2GRAY)
    for ang in (-8,-4,0,4,8):
        M=cv2.getRotationMatrix2D((36,36),ang,1.0)
        rr=cv2.warpAffine(g,M,(72,72),borderMode=cv2.BORDER_REPLICATE)
        ee=cv2.Canny(cv2.createCLAHE(1.4,(8,8)).apply(rr),55,150).astype(np.float32)
        ee=(ee-ee.mean())/(ee.std()+1e-6)
        TF[label].append(ee)

def template_score(crop, label):
    if label not in TF or crop is None or crop.size == 0:
        return 0.0
    g=cv2.cvtColor(cv2.resize(crop,(72,72)),cv2.COLOR_BGR2GRAY)
    e=cv2.Canny(cv2.createCLAHE(1.4,(8,8)).apply(g),55,150).astype(np.float32)
    e=(e-e.mean())/(e.std()+1e-6)
    return max(float((e*t).mean()) for t in TF[label])

def evidence_score(conf, family, source, color, shape, tmpl):
    score=float(conf)
    if family=='p':
        score += 0.035  # primary already works well on this video
    if 'global' in source:
        score += 0.018
    score += 0.030*min(1.0,color*7.0)
    score += 0.022*shape
    score += 0.085*max(0.0,tmpl)
    return score

def parse_boxes(result, frame, source, family, offset=(0,0),
                inv=None, source_shape=None):
    """Parse YOLO result and map coordinates back to the original full frame."""
    H,W=frame.shape[:2]
    ox,oy=offset
    names=result.names
    out=[]
    if result.boxes is None:
        return out
    for b in result.boxes:
        x1,y1,x2,y2=map(float,b.xyxy[0].tolist())

        # For static primary, result is in the manually letterboxed square.
        if inv is not None:
            r,left,top=inv
            x1=(x1-left)/r; x2=(x2-left)/r
            y1=(y1-top)/r;  y2=(y2-top)/r
            ih,iw=source_shape
            x1=max(0,min(iw-1,x1)); x2=max(0,min(iw-1,x2))
            y1=max(0,min(ih-1,y1)); y2=max(0,min(ih-1,y2))

        bb=(int(round(x1+ox)),int(round(y1+oy)),
            int(round(x2+ox)),int(round(y2+oy)))
        bb=(max(0,bb[0]),max(0,bb[1]),min(W-1,bb[2]),min(H-1,bb[3]))
        if not plausible(bb,W,H):
            continue

        cls=int(b.cls[0])
        raw=str(names.get(cls,cls) if isinstance(names,dict) else names[cls])
        if family=='s' and raw in EXCLUDE:
            continue
        label=primary_label(raw) if family=='p' else secondary_label(raw)
        conf=float(b.conf[0])

        crop=frame[bb[1]:bb[3],bb[0]:bb[2]]
        cs=color_support(crop)
        ss=shape_support(crop) if conf < 0.55 else 0.0
        ts=template_score(crop,label) if conf < 0.48 else 0.0
        score=evidence_score(conf,family,source,cs,ss,ts)
        out.append(Det(bb,label,conf,source,family,cs,ss,ts,score))
    return out

def infer_primary(img, frame, source, conf, offset=(0,0)):
    # STATIC SHAPE RULE: always exactly 640x640.
    sq,r,left,top=letterbox_square(img,PRIMARY_640)
    result=primary.predict(
        sq, conf=conf, imgsz=PRIMARY_640, device=DEVICE,
        verbose=False, max_det=60
    )[0]
    return parse_boxes(
        result,frame,source,'p',offset=offset,
        inv=(r,left,top),source_shape=img.shape[:2]
    )

def infer_secondary(img, frame, source, conf, imgsz=704, offset=(0,0)):
    result=secondary.predict(
        img, conf=conf, imgsz=imgsz, device=DEVICE,
        verbose=False, max_det=70
    )[0]
    return parse_boxes(result,frame,source,'s',offset=offset)

def far_tiles(frame):
    """Two overlapping upper-road crops. ~1.7-1.9x effective zoom vs full frame."""
    H,W=frame.shape[:2]
    y1=int(H*0.05); y2=int(H*0.72)
    tw=int(W*0.56)
    return [
        (frame[y1:y2, 0:tw], (0,y1), 'far_left'),
        (frame[y1:y2, W-tw:W], (W-tw,y1), 'far_right'),
    ]


In [ ]:
# 4) Cross-pass merge + strict current-frame temporal stabilizer

def cross_pass_merge(dets):
    """Deduplicate same-object detections and suppress ambiguous class conflicts."""
    dets=sorted(dets,key=lambda d:d.score,reverse=True)
    kept=[]
    for d in dets:
        conflict_idx=None
        same_idx=None
        for i,k in enumerate(kept):
            ov=iou(d.box,k.box)
            if ov < 0.42:
                continue
            if d.label == k.label:
                same_idx=i
                break
            if ov >= 0.55:
                conflict_idx=i
                break

        if same_idx is not None:
            k=kept[same_idx]
            # Prefer stronger evidence; do not average boxes from radically different crops.
            if d.score > k.score:
                kept[same_idx]=d
            continue

        if conflict_idx is not None:
            k=kept[conflict_idx]
            margin=abs(d.score-k.score)
            if margin < 0.10:
                # Two models/passes strongly disagree on the same object -> suppress ambiguity.
                kept.pop(conflict_idx)
            elif d.score > k.score:
                kept[conflict_idx]=d
            continue

        kept.append(d)
    return kept

@dataclass
class Track:
    tid: int
    label: str
    box: tuple
    last_frame: int
    hits: deque = field(default_factory=lambda: deque(maxlen=6))
    mean_score: float = 0.0

class CurrentEvidenceStabilizer:
    """
    No extrapolation: output exists only when a REAL detection exists in current frame.
    History is used only to confirm label/object identity and smooth the current box.
    """
    def __init__(self):
        self.tracks={}
        self.next_id=1

    def _match(self,d):
        best=None; best_metric=999.0
        for tid,t in self.tracks.items():
            if t.label != d.label:
                continue
            ov=iou(t.box,d.box)
            cd=center_match(t.box,d.box)
            # IoU for medium/large objects; center fallback is important for 8-20px far signs.
            if ov >= 0.18:
                metric=1.0-ov
            elif cd <= 0.85:
                metric=1.0+cd
            else:
                continue
            if metric < best_metric:
                best_metric=metric; best=tid
        return best

    def update(self,dets,frame_idx):
        out=[]
        used=set()
        for d in sorted(dets,key=lambda z:z.score,reverse=True):
            tid=self._match(d)
            if tid is None or tid in used:
                tid=self.next_id; self.next_id+=1
                self.tracks[tid]=Track(tid,d.label,d.box,frame_idx)
            t=self.tracks[tid]
            used.add(tid)

            # Faster response for large motion; stronger smoothing for small jitter.
            ov=iou(t.box,d.box)
            alpha=0.78 if ov < 0.20 else 0.62
            sm=tuple(int(round(alpha*n+(1-alpha)*o)) for n,o in zip(d.box,t.box))
            t.box=sm; t.last_frame=frame_idx
            t.hits.append((frame_idx,d.score,d.conf,d.source,d.tmpl,d.color,d.shape))
            vals=[x[1] for x in t.hits]
            t.mean_score=float(np.mean(vals)) if vals else d.score

            recent=[x for x in t.hits if frame_idx-x[0] <= 4]
            recent3=[x for x in t.hits if frame_idx-x[0] <= 5]

            # Strong current evidence can appear immediately.
            strong = d.conf >= (0.62 if 'global' in d.source else 0.70)

            # Normal detections: at least 2 real sightings in a short window.
            stable = len(recent) >= 2 and t.mean_score >= 0.255

            # Very weak / rescue evidence requires 3 sightings OR template+color/shape support.
            weak = d.conf < 0.22 or 'rescue' in d.source
            verified_weak = (
                len(recent3) >= 3 and t.mean_score >= 0.235 and
                (d.tmpl >= 0.10 or d.color >= 0.035 or d.shape >= 0.55)
            )

            show = strong or (stable and not weak) or verified_weak
            if show:
                d.box=sm
                out.append((tid,d))

        # Keep short history for matching only, but NEVER draw absent tracks.
        stale=[tid for tid,t in self.tracks.items() if frame_idx-t.last_frame > 8]
        for tid in stale:
            del self.tracks[tid]
        return out

stabilizer=CurrentEvidenceStabilizer()


In [ ]:
# 5) Optional DIP rescue proposals
# These propose WHERE to zoom. YOLO must still confirm the crop; DIP/templates never assign the class.

def dip_proposals(frame, existing):
    H,W=frame.shape[:2]
    y2=int(H*0.76)
    roi=frame[:y2]
    hsv=cv2.cvtColor(roi,cv2.COLOR_BGR2HSV)

    masks=[
        cv2.inRange(hsv,(0,75,45),(12,255,255)),
        cv2.inRange(hsv,(165,75,45),(180,255,255)),
        cv2.inRange(hsv,(90,60,40),(138,255,255)),
        cv2.inRange(hsv,(15,65,50),(40,255,255)),
    ]
    m=masks[0]
    for z in masks[1:]:
        m=cv2.bitwise_or(m,z)
    m=cv2.morphologyEx(m,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))

    cnts,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cand=[]
    for c in cnts:
        a=cv2.contourArea(c)
        if not 20 < a < 8500:
            continue
        x,y,w,h=cv2.boundingRect(c)
        if min(w,h)<5 or max(w,h)>155 or not 0.42 < w/max(1,h) < 1.85:
            continue
        per=cv2.arcLength(c,True)
        circ=4*math.pi*a/(per*per+1e-6)
        verts=len(cv2.approxPolyDP(c,0.04*per,True))
        if circ < 0.32 and not 3 <= verts <= 8:
            continue

        cx=x+w/2; cy=y+h/2
        side=int(max(56,max(w,h)*3.0))
        bb=(max(0,int(cx-side/2)),max(0,int(cy-side/2)),
            min(W,int(cx+side/2)),min(y2,int(cy+side/2)))
        if any(iou(bb,d.box) > 0.22 for d in existing):
            continue
        ratio=np.count_nonzero(m[y:y+h,x:x+w])/max(1,w*h)
        priority=ratio + min(1.0,a/1800)
        cand.append((priority,bb))

    cand.sort(reverse=True)
    out=[]
    for _,bb in cand:
        if any(iou(bb,k)>0.30 for k in out):
            continue
        out.append(bb)
        if len(out)>=1:  # strict runtime/false-positive cap
            break
    return out

def rescue_infer(frame, bb):
    x1,y1,x2,y2=bb
    crop=frame[y1:y2,x1:x2]
    if crop.size==0:
        return []
    if PRIMARY_OK:
        ds=infer_primary(adaptive_enhance(crop),frame,'p_rescue',0.10,(x1,y1))
    else:
        ds=infer_secondary(adaptive_enhance(crop),frame,'s_rescue',0.13,640,(x1,y1))

    # Candidate must land near the proposal center and have supporting appearance evidence.
    cx=(x1+x2)/2; cy=(y1+y2)/2
    bw=max(1,x2-x1); bh=max(1,y2-y1)
    out=[]
    for d in ds:
        dx=((d.box[0]+d.box[2])/2-cx)/bw
        dy=((d.box[1]+d.box[3])/2-cy)/bh
        if abs(dx)>0.32 or abs(dy)>0.32:
            continue
        if d.color < 0.025 and d.shape < 0.45 and d.tmpl < 0.08:
            continue
        out.append(d)
    return out


In [ ]:
# 6) SMOKE TEST — validates model/input shapes before processing all 3087 frames

cap=cv2.VideoCapture(str(VIDEO))
ok, smoke=cap.read()
cap.release()
assert ok, 'Could not read first video frame'

PRIMARY_OK=True
try:
    a=infer_primary(smoke,smoke,'p_global_smoke',0.22)
    b=[]
    for tile,off,name in far_tiles(smoke):
        b += infer_primary(adaptive_enhance(tile),smoke,'p_'+name+'_smoke',0.13,off)
    print(f'Primary static-640 smoke test: OK ({len(a)} global, {len(b)} far detections)')
except Exception as e:
    PRIMARY_OK=False
    print('WARNING: primary TorchScript failed even at exact 640x640.')
    print('Pipeline will automatically continue with the secondary .pt model only.')
    print(type(e).__name__, ':', str(e)[:500])

try:
    s=infer_secondary(smoke,smoke,'s_global_smoke',0.22,704)
    ft,fo,fn=far_tiles(smoke)[0]
    sf=infer_secondary(adaptive_enhance(ft),smoke,'s_far_smoke',0.16,800,fo)
    print(f'Secondary YOLO11s smoke test: OK ({len(s)} global, {len(sf)} far detections)')
except Exception as e:
    raise RuntimeError(f'Secondary model smoke test failed: {e}') from e

torch.cuda.empty_cache()
print('Smoke test passed. Safe to process the full video.')


In [ ]:
# 7) PROCESS VIDEO — global + far zoom + conservative rescue

OUT=OUTPUTS/'video1_result.mp4'
CSV=OUTPUTS/'video1_result.csv'
TEMP=OUTPUTS/'video1_result_temp.mp4'

for p in (OUT,CSV,TEMP):
    try: p.unlink()
    except FileNotFoundError: pass

cap=cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), f'Cannot open {VIDEO}'
fps=float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer=cv2.VideoWriter(str(TEMP),cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
assert writer.isOpened(), 'Cannot create temporary output video'

fields=['frame','time_sec','track_id','label','confidence','score',
        'source','x1','y1','x2','y2','color_support','shape_support','template_score']

def draw_label(frame,box,text):
    x1,y1,x2,y2=map(int,box)
    color=(0,165,255)
    cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
    font=cv2.FONT_HERSHEY_SIMPLEX
    scale=0.52 if H<=1080 else 0.60
    thick=2
    (tw,th),base=cv2.getTextSize(text,font,scale,thick)
    pad=5
    yy=max(0,y1-th-base-pad*2)
    xx=max(0,min(W-tw-pad*2-1,x1))
    cv2.rectangle(frame,(xx,yy),(xx+tw+pad*2,yy+th+base+pad*2),color,-1)
    cv2.putText(frame,text,(xx+pad,yy+th+pad),font,scale,(255,255,255),thick,cv2.LINE_AA)

counts=defaultdict(int)
source_counts=defaultdict(int)

with open(CSV,'w',newline='',encoding='utf-8') as f:
    log=csv.DictWriter(f,fieldnames=fields)
    log.writeheader()
    bar=tqdm(total=total or None,desc='Traffic signs: robust global + far zoom',unit='frame')
    frame_idx=0

    while True:
        ok,frame=cap.read()
        if not ok:
            break

        dets=[]

        # A) Global, every frame: preserves the nearby-sign quality.
        if PRIMARY_OK:
            dets += infer_primary(frame,frame,'p_global',0.22)
        dets += infer_secondary(frame,frame,'s_global',0.24,704)

        # B) Far zoom, every frame for the primary so far signs do not blink.
        tiles=far_tiles(frame)
        if PRIMARY_OK:
            for tile,off,name in tiles:
                dets += infer_primary(adaptive_enhance(tile),frame,'p_'+name,0.13,off)
        else:
            # Fallback if static TorchScript is incompatible with future runtime.
            for tile,off,name in tiles:
                dets += infer_secondary(adaptive_enhance(tile),frame,'s_'+name,0.15,800,off)

        # C) Secondary far pass every 2 frames: complementary 82-class evidence.
        if frame_idx % 2 == 0:
            # Alternate left/right to reduce compute while covering both sides over time.
            tile,off,name=tiles[(frame_idx//2)%2]
            dets += infer_secondary(adaptive_enhance(tile),frame,'s_'+name,0.16,800,off)

        merged=cross_pass_merge(dets)

        # D) One strict DIP rescue crop every 3 frames, only outside existing YOLO boxes.
        if frame_idx % 3 == 0:
            props=dip_proposals(frame,merged)
            for bb in props:
                merged += rescue_infer(frame,bb)
            merged=cross_pass_merge(merged)

        # E) Current-frame-only temporal confirmation + EMA.
        shown=stabilizer.update(merged,frame_idx)

        for tid,d in shown:
            label=f'{d.label} {d.conf:.2f}'
            draw_label(frame,d.box,label)
            x1,y1,x2,y2=d.box
            log.writerow({
                'frame':frame_idx,
                'time_sec':round(frame_idx/fps,3),
                'track_id':tid,
                'label':d.label,
                'confidence':round(d.conf,4),
                'score':round(d.score,4),
                'source':d.source,
                'x1':x1,'y1':y1,'x2':x2,'y2':y2,
                'color_support':round(d.color,4),
                'shape_support':round(d.shape,4),
                'template_score':round(d.tmpl,4),
            })
            counts[d.label]+=1
            source_counts[d.source]+=1

        writer.write(frame)
        frame_idx+=1
        bar.update(1)

    bar.close()

cap.release()
writer.release()

# H.264 + yuv420p + optional original audio, browser/Drive compatible.
video_args=['-c:v','libx264','-preset','veryfast','-crf','22',
            '-pix_fmt','yuv420p','-tag:v','avc1','-movflags','+faststart']
cmd=[
    'ffmpeg','-y','-loglevel','error',
    '-i',str(TEMP),'-i',str(VIDEO),
    '-map','0:v:0','-map','1:a?',
    *video_args,'-c:a','aac','-b:a','128k','-shortest',str(OUT)
]
try:
    subprocess.run(cmd,check=True)
except subprocess.CalledProcessError:
    subprocess.run([
        'ffmpeg','-y','-loglevel','error','-i',str(TEMP),
        *video_args,'-an',str(OUT)
    ],check=True)

try: TEMP.unlink()
except FileNotFoundError: pass

assert OUT.exists() and OUT.stat().st_size>0
print('DONE')
print('Output:',OUT)
print('CSV:',CSV)
print('Frames:',frame_idx,'| FPS:',fps)
print('Primary enabled:',PRIMARY_OK)
print('Most frequent labels:',dict(sorted(counts.items(),key=lambda kv:kv[1],reverse=True)[:12]))
print('Sources:',dict(sorted(source_counts.items(),key=lambda kv:kv[1],reverse=True)))


In [ ]:
# 8) SHOW RESULT DIRECTLY IN COLAB
from IPython.display import Video, display

PREVIEW=Path('/content/video1_result_preview.mp4')
try: PREVIEW.unlink()
except FileNotFoundError: pass

subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',str(OUT),
    '-vf','scale=960:-2',
    '-c:v','libx264','-preset','veryfast','-crf','29',
    '-pix_fmt','yuv420p','-tag:v','avc1','-movflags','+faststart',
    '-an',str(PREVIEW)
],check=True)

print('Full result on Drive:',OUT)
display(Video(str(PREVIEW),embed=True,width=960,html_attributes='controls'))


## After a Colab disconnect

The model weights and template cache are already in `MyDrive/DIP/models/`.

Open this same notebook again and use **Runtime → Run all**. Hugging Face will reuse the files already present in Drive; **there is no training step**.
